# NYC Taxi Fare Prediction — Data Exploration & Cleaning

This notebook covers data loading, data-quality checks, exploratory analysis, cleaning decisions, and final feature engineering.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Load the raw dataset

In [ ]:
df = pd.read_parquet("../data/raw/yellow_tripdata_2025-01.parquet")

df.head()


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()


## 3. Initial data inspection

In [ ]:
df.describe()


In [ ]:
df.isna().sum()


In [ ]:
df.nunique()


## 4. Data-quality checks

In [ ]:
print("Negative fares:", (df["fare_amount"] < 0).sum())
print("Zero fares:", (df["fare_amount"] == 0).sum())


In [ ]:
print("Zero distance:", (df["trip_distance"] == 0).sum())
print("Negative distance:", (df["trip_distance"] < 0).sum())


In [ ]:
duration = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]

print("Negative duration:", (duration < pd.Timedelta(0)).sum())
print("Zero duration:", (duration == pd.Timedelta(0)).sum())
print("Max duration:", duration.max())


## 5. Create trip duration

In [ ]:
duration = (
    df["tpep_dropoff_datetime"]
    - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration"] = duration


In [ ]:
df[[
    "fare_amount",
    "trip_distance",
    "trip_duration"
]].quantile([
    0,
    0.001,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    0.999,
    1.0
])


## 6. Inspect suspicious observations

In [ ]:
df.loc[
    df["fare_amount"] < 0,
    ["fare_amount", "trip_distance", "trip_duration",
     "payment_type", "RatecodeID"]
].head(20)


In [ ]:
df.loc[
    df["trip_distance"] == 0,
    ["fare_amount", "trip_distance", "trip_duration",
     "payment_type", "RatecodeID"]
].head(20)


In [ ]:
df.nlargest(
    20,
    "trip_duration"
)[[
    "fare_amount",
    "trip_distance",
    "trip_duration",
    "payment_type",
    "RatecodeID"
]]


## 7. Initial cleaning

The first cleaning pass removes non-positive fares and non-positive trip durations. We also restrict trip distance and duration to plausible ranges based on the exploratory analysis.

In [ ]:
df_clean = df[df["fare_amount"] > 0].copy()

df_clean = df_clean[
    df_clean["trip_duration"] > 0
].copy()

df_clean = df_clean[
    (df_clean["trip_distance"] > 0) &
    (df_clean["trip_distance"] <= 100) &
    (df_clean["trip_duration"] <= 180)
].copy()


In [ ]:
print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))
print("Percentage removed:", (len(df) - len(df_clean)) / len(df) * 100)


In [ ]:
df_clean[[
    "fare_amount",
    "trip_distance",
    "trip_duration"
]].describe()


## 8. Distributions after initial cleaning

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_clean["fare_amount"], bins=100, ax=axes[0])
axes[0].set_title("Fare Distribution")
axes[0].set_xlim(0, 100)

sns.histplot(df_clean["trip_distance"], bins=100, ax=axes[1])
axes[1].set_title("Trip Distance")
axes[1].set_xlim(0, 30)

sns.histplot(df_clean["trip_duration"], bins=100, ax=axes[2])
axes[2].set_title("Trip Duration")
axes[2].set_xlim(0, 120)

plt.tight_layout()
plt.show()


## 9. Fare outlier investigation

In [ ]:
df_clean["fare_amount"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999, 0.9999]
)


In [ ]:
df_clean = df_clean[
    df_clean["fare_amount"] <= 300
].copy()

print("Maximum fare:", df_clean["fare_amount"].max())
print("Fares above 300:", (df_clean["fare_amount"] > 300).sum())


In [ ]:
print(df_clean.loc[df_clean["fare_amount"].idxmax(), [
    "fare_amount",
    "trip_distance",
    "trip_duration"
]])


## 10. Time-based features

In [ ]:
df_clean["pickup_hour"] = df_clean["tpep_pickup_datetime"].dt.hour
df_clean["pickup_day"] = df_clean["tpep_pickup_datetime"].dt.dayofweek
df_clean["pickup_day_of_month"] = df_clean["tpep_pickup_datetime"].dt.day
df_clean["pickup_month"] = df_clean["tpep_pickup_datetime"].dt.month

df_clean["is_weekend"] = (
    df_clean["pickup_day"] >= 5
).astype(int)

df_clean["is_rush_hour"] = (
    df_clean["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19])
).astype(int)


## 11. Investigate distance-duration inconsistencies

In [ ]:
df_clean[
    (df_clean["trip_distance"] > 50) &
    (df_clean["trip_duration"] < 30)
][[
    "fare_amount",
    "trip_distance",
    "trip_duration",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type"
]].sort_values("trip_distance", ascending=False).head(20)


## 12. Average-speed feature

In [ ]:
df_clean["avg_speed_mph"] = (
    df_clean["trip_distance"] /
    df_clean["trip_duration"] * 60
)

df_clean["avg_speed_mph"].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
)


In [ ]:
df_clean.nlargest(
    20,
    "avg_speed_mph"
)[[
    "fare_amount",
    "trip_distance",
    "trip_duration",
    "avg_speed_mph",
    "RatecodeID",
    "payment_type"
]]


In [ ]:
df_clean.nsmallest(
    20,
    "avg_speed_mph"
)[[
    "fare_amount",
    "trip_distance",
    "trip_duration",
    "avg_speed_mph",
    "RatecodeID",
    "payment_type"
]]


## 13. Remove implausible average speeds

In [ ]:
before = len(df_clean)

df_clean = df_clean[
    (df_clean["avg_speed_mph"] >= 1) &
    (df_clean["avg_speed_mph"] <= 70)
].copy()

after = len(df_clean)

print("Removed:", before - after)
print("Percentage removed:", (before - after) / before * 100)


In [ ]:
df_clean[[
    "fare_amount",
    "trip_distance",
    "trip_duration",
    "avg_speed_mph"
]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
)


In [ ]:
df_clean[[
    "fare_amount",
    "trip_distance",
    "trip_duration",
    "avg_speed_mph"
]].hist(
    bins=50,
    figsize=(12, 8)
)

plt.tight_layout()
plt.show()


## 14. Final cleaned dataset

In [ ]:
print("Final shape:", df_clean.shape)

print("\nMissing values in model-related columns:")
print(df_clean[[
    "trip_distance",
    "trip_duration",
    "avg_speed_mph",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type"
]].isna().sum())


In [ ]:
print("Fare range:", df_clean["fare_amount"].min(), "to", df_clean["fare_amount"].max())
print("Distance range:", df_clean["trip_distance"].min(), "to", df_clean["trip_distance"].max())
print("Duration range:", df_clean["trip_duration"].min(), "to", df_clean["trip_duration"].max())
print("Average-speed range:", df_clean["avg_speed_mph"].min(), "to", df_clean["avg_speed_mph"].max())


## 15. Save processed data

The modeling notebook will start from this processed dataset so the cleaning work does not have to be repeated.

In [ ]:
processed_path = "../data/processed/yellow_tripdata_2025-01_clean.parquet"

df_clean.to_parquet(processed_path, index=False)

print(f"Saved processed dataset to: {processed_path}")
